# Relatedness in a human GWAS cohort

A GWAS treats individuals as independent samples. If some of them are close relatives that
assumption breaks: relatives share long stretches of genome, so they share genotypes at
many SNPs at once, and the test statistics come out too large.

So before running an association study you check who is related to whom. This is a short
exercise that does exactly that, on the same case/control cohort used in the GWAS
exercises.

### How PLINK estimates relatedness

For every **pair** of individuals PLINK estimates how much of their genome they share
*identical by descent* (IBD) - inherited from a common ancestor. It reports three numbers
that sum to one:

| | meaning |
| --- | --- |
| **Z0** | fraction of the genome where they share **0** alleles IBD |
| **Z1** | fraction where they share **1** allele IBD |
| **Z2** | fraction where they share **2** alleles IBD |

and it summarises them as

$$\hat{\pi} = \texttt{PI\_HAT} = Z_2 + \tfrac{1}{2}Z_1$$

which is the expected proportion of alleles shared IBD - twice the kinship coefficient.
The textbook expectations are:

| relationship | Z0 | Z1 | Z2 | PI_HAT |
| --- | --- | --- | --- | --- |
| the same person (or identical twins) | 0 | 0 | 1 | 1 |
| parent - offspring | 0 | 1 | 0 | 0.5 |
| full siblings | 0.25 | 0.5 | 0.25 | 0.5 |
| half sibs, uncle - nephew, grandparent | 0.5 | 0.5 | 0 | 0.25 |
| first cousins | 0.75 | 0.25 | 0 | 0.125 |
| unrelated | 1 | 0 | 0 | 0 |

> **An important assumption.** This estimator works from allele frequencies estimated in
> your sample, and it assumes everybody is drawn from **one homogeneous population**. If
> the sample instead contains several populations, individuals from the same population
> look "related" to each other simply because they share ancestry, and PI_HAT is inflated.
> That is fine for this data set, which is a single population - but it is the reason the
> same analysis on a structured sample needs a different method.

## 1. Setup

In [ ]:
#############################################################
# ALL PATHS ARE SET HERE
# If the data moves, this is the ONLY cell you need to change.
#############################################################

# where the shared data lives
DATA=/course/data/current_data/gwas_human

# where you will do the exercise
WORK=$HOME/relatedness_human
mkdir -p $WORK
cd $WORK

cat > $WORK/env.sh <<EOF
export DATA=$DATA
export WORK=$WORK
cd $WORK
EOF

echo "working folder: $(pwd)"
echo
echo "the data is a PLINK binary fileset:"
ls -l $DATA

### The data

The genotypes are in PLINK's binary format: `.bed` holds the genotypes, `.bim` the SNPs
and `.fam` the individuals.

In [ ]:
source ~/relatedness_human/env.sh

echo "individuals: $(wc -l < $DATA/gwa.fam)"
echo "SNPs:        $(wc -l < $DATA/gwa.bim)"

echo
echo "--- the last column of the .fam is the phenotype (1 = control, 2 = case) ---"
awk '{print $6}' $DATA/gwa.fam | sort | uniq -c

echo
echo "--- first lines of the .fam ---"
head -n 3 $DATA/gwa.fam

**Questions**

 - How many individuals and how many SNPs are there?
 - How many pairs of individuals does that make? (that is how many relatedness estimates
   we are about to get)

## 2. Estimate relatedness

`--genome` does the pairwise IBD estimation. `--autosome` keeps only the autosomes, since
the sex chromosomes do not behave like the rest of the genome, and `--maf 0.05` drops rare
SNPs, whose frequencies are too poorly estimated to be useful here.

This takes a minute or two.

In [ ]:
source ~/relatedness_human/env.sh
# ~1-2 min

plink --bfile $DATA/gwa --genome --autosome --maf 0.05 --out gwa_ibd

echo
echo "--- created files ---"
ls -l gwa_ibd.genome

Have a look at the output. One line per pair of individuals.

In [ ]:
source ~/relatedness_human/env.sh

echo "--- the columns ---"
head -n 1 gwa_ibd.genome

echo
echo "--- the first few pairs ---"
head -n 4 gwa_ibd.genome

echo
echo "--- number of pairs ---"
echo $(( $(wc -l < gwa_ibd.genome) - 1 ))

## 3. Plot it

The standard way to look at this is Z1 against Z0, with one point per pair. Each
relationship from the table above sits at its own place in that square, so relatives stand
out as points away from the bottom-right corner.

In [ ]:
setwd(path.expand("~/relatedness_human"))

g <- read.table("gwa_ibd.genome", header = TRUE)
cat("pairs read:", nrow(g), "\n")

# PLINK writes 'nan' for some pairs - keep track of those separately
ok <- !is.na(g$PI_HAT)
cat("pairs with an estimate:", sum(ok), "  pairs with NaN:", sum(!ok), "\n")

plot(g$Z0[ok], g$Z1[ok],
     pch = 16, col = rgb(0, 0, 0, 0.25), cex = 0.7,
     xlim = c(0, 1), ylim = c(0, 1),
     xlab = "Z0   (fraction of the genome sharing 0 alleles IBD)",
     ylab = "Z1   (fraction sharing 1 allele IBD)",
     main = "IBD sharing, one point per pair of individuals")

# where the textbook relationships sit
expected <- data.frame(
    Z0  = c(0,           0,                  0.25,         0.5,        0.75,            1),
    Z1  = c(0,           1,                  0.5,          0.5,        0.25,            0),
    lab = c("duplicate", "parent-offspring", "full sibs",  "half sibs", "first cousins", "unrelated")
)
points(expected$Z0, expected$Z1, pch = 4, col = "red", cex = 1.6, lwd = 2)
text(expected$Z0, expected$Z1, expected$lab,
     pos = c(4, 4, 4, 4, 4, 2), col = "red", cex = 0.75)

**Questions**
 - What is the largest PI_HAT in this cohort, and would you call that pair related?
 - Some pairs are `nan`. Which individuals are involved, and what do they have in common?

And the distribution of `PI_HAT` itself. The dashed lines mark first-degree (0.5),
second-degree (0.25) and third-degree (0.125) relatives.

In [ ]:
setwd(path.expand("~/relatedness_human"))

hist(g$PI_HAT[ok], breaks = 60, col = "steelblue", border = "white",
     xlim = c(0, 1),
     xlab = "PI_HAT", ylab = "number of pairs",
     main = "Relatedness across all pairs")
abline(v = c(0.5, 0.25, 0.125), col = "red", lty = 2)
text(c(0.5, 0.25, 0.125), par("usr")[4]*0.8,
     c("1st degree", "2nd", "3rd"), col = "red", pos = 4, cex = 0.8)

cat("largest PI_HAT in the data:", max(g$PI_HAT, na.rm = TRUE), "\n\n")
cat("pairs above each threshold:\n")
for (t in c(0.5, 0.25, 0.125, 0.08)) {
    cat(sprintf("  PI_HAT > %.3f : %d pairs\n", t, sum(g$PI_HAT > t, na.rm = TRUE)))
}

**Questions**

 - Where do almost all the pairs sit in the Z0/Z1 plot, and which relationship is that?
 - What is the largest `PI_HAT`? Is there **any** pair you would call first- or
   second-degree relatives?
 - Given the answer, was it safe to run a GWAS on this cohort without removing anyone for
   relatedness?
 - The histogram is not centred on exactly zero. Why would unrelated pairs still show a
   little bit of apparent sharing?

## 4. The pairs that came out as NaN

The plot above quietly dropped some pairs, because PLINK could not estimate them and wrote
`nan`. That is worth chasing rather than ignoring - it is telling us something about the
data.

In [ ]:
source ~/relatedness_human/env.sh

echo "--- how many pairs failed? ---"
awk 'NR>1 && $10=="nan"' gwa_ibd.genome | wc -l

echo
echo "--- which individuals are involved? ---"
# NB: in this fileset the informative label is the FID (column 1 / 3);
# the IID is just "1" for everybody
awk 'NR>1 && $10=="nan" {print $1; print $3}' gwa_ibd.genome | sort -u

A pair can only fail if there is too little data. Let us measure how much genotype data
each individual actually has, with `--missing`.

In [ ]:
source ~/relatedness_human/env.sh

plink --bfile $DATA/gwa --missing --out gwa_miss > /dev/null 2>&1

echo "--- the per-individual missingness file ---"
head -n 1 gwa_miss.imiss

echo
echo "--- the 10 individuals with the most missing data (F_MISS = fraction missing) ---"
awk 'NR>1 {print $6"\t"$1}' gwa_miss.imiss | sort -gr | head -n 10

echo
echo "--- how many individuals are missing more than 20% of their genotypes? ---"
awk 'NR>1 && $6>0.2' gwa_miss.imiss | wc -l

In [ ]:
setwd(path.expand("~/relatedness_human"))

m <- read.table("gwa_miss.imiss", header = TRUE)
m <- m[order(-m$F_MISS), ]

plot(m$F_MISS, pch = 16, cex = 0.8,
     col = ifelse(m$F_MISS > 0.2, "red", "grey40"),
     xlab = "individual, sorted by missingness",
     ylab = "fraction of genotypes missing",
     main = "How much data does each individual have?")
abline(h = 0.2, lty = 2)
legend("topright", c("more than 20% missing", "kept"),
       pch = 16, col = c("red", "grey40"), bty = "n")

cat("individuals above 20% missing:", sum(m$F_MISS > 0.2), "\n")

**Questions**

 - Compare the red individuals with the list of individuals in the failed pairs. Are they
   the same people?
 - Roughly what fraction of their genotypes are those individuals missing?
 - Why does that make the IBD estimate impossible for a **pair** of them? (think about how
   many SNPs are left where *both* members of the pair have a genotype)

## 5. Redo it on the individuals with enough data

`--mind 0.2` drops individuals missing more than 20% of their genotypes, before anything
else runs.

In [ ]:
source ~/relatedness_human/env.sh
# ~1-2 min

plink --bfile $DATA/gwa --genome --autosome --maf 0.05 --mind 0.2 --out gwa_ibd_clean

echo
echo "--- how many pairs failed this time? ---"
awk 'NR>1 && $10=="nan"' gwa_ibd_clean.genome | wc -l

In [ ]:
setwd(path.expand("~/relatedness_human"))

gc2 <- read.table("gwa_ibd_clean.genome", header = TRUE)

cat("before  --mind 0.2 :", nrow(g),   "pairs,", sum(is.na(g$PI_HAT)),   "NaN\n")
cat("after   --mind 0.2 :", nrow(gc2), "pairs,", sum(is.na(gc2$PI_HAT)), "NaN\n\n")
cat("largest PI_HAT before:", max(g$PI_HAT,   na.rm = TRUE), "\n")
cat("largest PI_HAT after: ", max(gc2$PI_HAT, na.rm = TRUE), "\n")

par(mfrow = c(1, 2))
hist(g$PI_HAT[ok], breaks = 60, col = "grey70", border = "white",
     xlab = "PI_HAT", main = "all individuals")
hist(gc2$PI_HAT, breaks = 60, col = "steelblue", border = "white",
     xlab = "PI_HAT", main = "missingness > 20% removed")
par(mfrow = c(1, 1))

**Questions**

 - Did removing those individuals change the conclusion about relatedness in this cohort?
 - The number of pairs dropped a lot. Why does removing 15 individuals remove so many
   pairs?
 - In a real study, would you drop those individuals, or try to get better data for them?

### And finally, back to the assumption

The note at the top said this estimator assumes a single homogeneous population.

 - If this cohort had actually been a mix of, say, European and East Asian individuals,
   what would the Z0/Z1 plot have looked like, and which pairs would have been wrongly
   flagged as relatives?
 - Relatedness and population structure are two different problems that produce a similar
   symptom in a GWAS. How would you tell them apart?

### Run the cell below to take the quiz

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/relatedness_diversity/quiz/relatedness_human.json")
